# Evaluation RAG #
### Text2SQL step

In [26]:
from llm import llm
from models import state
import pandas as pd
from langgraph.graph import StateGraph, START, END # Start node and End node

## Import related node #

In [27]:
from nodes.schema import schema
from nodes.text2sql import Text2SQLNode
from nodes.executeSQL import executeSQLNode

## Build Unit Graph 
### test Text2SQL result Only

In [ ]:
from sqlalchemy import text
from db import engine
import json

# executeSQLNode routes (via Command) to "verify_correctness" on success and back
# to "text2sql" on a failed query. This partial flow only evaluates up to execute_sql,
# so we register a tiny pass-through "verify_correctness" node that just ends — that
# lets executeSQL's goto resolve and the graph compile.
def dummy_node(state):
    return {}               # pass throught empty parameter return value 

def build_eval_graph():
    g = StateGraph(state)
    g.add_node("schema", schema)                       # gate: is question DB-related?
    g.add_node("text2sql", Text2SQLNode)               # NL question -> SQL
    g.add_node("executeSQL", executeSQLNode)           # run SQL (has a self-retry loop)
    g.add_node("verify_correctness", dummy_node)       # stub terminal -> END


    g.add_edge(START, "schema")                        # connect to each other #
    g.add_edge("text2sql", "executeSQL")               # text2sql returns a plain dict
    g.add_edge("verify_correctness", END)              # pass throught empty node 
    # schema & executeSQL route themselves via Command(goto=...)
    return g.compile()

eval_graph = build_eval_graph()


# ── helpers ───────────────────────────────────────────────────────────────────
def run_raw_sql(sql: str):
    """Execute a SQL string, return (rows_as_tuples, error_or_None)."""
    if not sql:
        return None, "empty sql"
    try:
        with engine.connect() as c:
            return [tuple(r) for r in c.execute(text(sql)).fetchall()], None
    except Exception as e:
        return None, str(e)

def normalize(rows):
    """Order-independent, type-loose key for comparing two result sets."""
    if rows is None:
        return None
    return sorted(str(tuple(r)) for r in rows)

def same_result(a, b):
    na, nb = normalize(a), normalize(b)
    return na is not None and na == nb


# ── Ground-truth cases: input question + expectation SQL ──────────────────────
# Covers the classicmodels relations:
#   productlines→products→orderdetails→orders→customers→payments ; employees→offices ;
#   customers→employees(salesRep). All 15 verified against the live DB.
#   (expectation OUTPUT is produced by executing expectation_sql — see the eval loop.)
TEST_CASES = [
    # -- single table --
    {"input": "How many customers are there?",
     "expectation_sql": "SELECT COUNT(customernumber) FROM customers;"},
    {"input": "List all offices with their city and country.",
     "expectation_sql": "SELECT officecode, city, country FROM offices ORDER BY officecode;"},
    {"input": "How many orders are there for each status?",
     "expectation_sql": "SELECT status, COUNT(ordernumber) AS orders FROM orders GROUP BY status ORDER BY orders DESC;"},

    # -- one join --
    {"input": "How many employees work in each office city?",
     "expectation_sql": "SELECT o.city, COUNT(e.employeenumber) AS employees FROM offices o "
                        "JOIN employees e ON e.officecode=o.officecode GROUP BY o.city ORDER BY employees DESC;"},
    {"input": "How many products are in each product line?",
     "expectation_sql": "SELECT productline, COUNT(productcode) AS products FROM products "
                        "GROUP BY productline ORDER BY products DESC;"},
    {"input": "How many customers does each sales rep handle?",
     "expectation_sql": "SELECT e.firstname||' '||e.lastname AS rep, COUNT(c.customernumber) AS customers "
                        "FROM employees e JOIN customers c ON c.salesrepemployeenumber=e.employeenumber "
                        "GROUP BY rep ORDER BY customers DESC;"},
    {"input": "Top 5 customers by total payments.",
     "expectation_sql": "SELECT c.customername, SUM(p.amount) AS total_paid FROM customers c "
                        "JOIN payments p ON p.customernumber=c.customernumber "
                        "GROUP BY c.customername ORDER BY total_paid DESC LIMIT 5;"},

    # -- aggregation + date --
    {"input": "What is the total sales revenue in 2004?",
     "expectation_sql": "SELECT SUM(od.quantityordered*od.priceeach) AS revenue FROM orderdetails od "
                        "JOIN orders o ON o.ordernumber=od.ordernumber WHERE EXTRACT(YEAR FROM o.orderdate)=2004;"},
    {"input": "Monthly sales revenue in 2004.",
     "expectation_sql": "SELECT TO_CHAR(o.orderdate,'YYYY-MM') AS month, SUM(od.quantityordered*od.priceeach) AS revenue "
                        "FROM orders o JOIN orderdetails od ON od.ordernumber=o.ordernumber "
                        "WHERE EXTRACT(YEAR FROM o.orderdate)=2004 GROUP BY month ORDER BY month;"},

    # -- multi-join --
    {"input": "Total sales revenue by product line in 2004.",
     "expectation_sql": "SELECT p.productline, SUM(od.quantityordered*od.priceeach) AS revenue FROM orderdetails od "
                        "JOIN orders o ON o.ordernumber=od.ordernumber JOIN products p ON p.productcode=od.productcode "
                        "WHERE EXTRACT(YEAR FROM o.orderdate)=2004 GROUP BY p.productline ORDER BY revenue DESC;"},
    {"input": "Top 5 best-selling products by revenue.",
     "expectation_sql": "SELECT p.productname, SUM(od.quantityordered*od.priceeach) AS revenue FROM orderdetails od "
                        "JOIN products p ON p.productcode=od.productcode GROUP BY p.productname "
                        "ORDER BY revenue DESC LIMIT 5;"},
    {"input": "Top 5 customers by total purchase amount.",
     "expectation_sql": "SELECT c.customername, SUM(od.quantityordered*od.priceeach) AS total FROM customers c "
                        "JOIN orders o ON o.customernumber=c.customernumber JOIN orderdetails od ON od.ordernumber=o.ordernumber "
                        "GROUP BY c.customername ORDER BY total DESC LIMIT 5;"},
    {"input": "Total revenue generated by each sales rep.",
     "expectation_sql": "SELECT e.firstname||' '||e.lastname AS rep, SUM(od.quantityordered*od.priceeach) AS revenue "
                        "FROM employees e JOIN customers c ON c.salesrepemployeenumber=e.employeenumber "
                        "JOIN orders o ON o.customernumber=c.customernumber "
                        "JOIN orderdetails od ON od.ordernumber=o.ordernumber GROUP BY rep ORDER BY revenue DESC;"},

    # -- CTE / business logic --
    {"input": "List customers who exceeded their credit limit based on total orders, and by how much.",
     "expectation_sql": "WITH t AS (SELECT o.customernumber, SUM(od.quantityordered*od.priceeach) AS total "
                        "FROM orders o JOIN orderdetails od ON od.ordernumber=o.ordernumber GROUP BY o.customernumber) "
                        "SELECT c.customername, t.total-c.creditlimit AS over_limit FROM customers c "
                        "JOIN t ON t.customernumber=c.customernumber WHERE t.total>c.creditlimit ORDER BY over_limit DESC;"},
    {"input": "List customers whose total orders exceed their total payments, sorted by highest outstanding balance.",
     "expectation_sql": "WITH billed AS (SELECT o.customernumber, SUM(od.quantityordered*od.priceeach) AS b FROM orders o "
                        "JOIN orderdetails od ON od.ordernumber=o.ordernumber GROUP BY o.customernumber), "
                        "paid AS (SELECT customernumber, SUM(amount) AS p FROM payments GROUP BY customernumber) "
                        "SELECT c.customername, billed.b-COALESCE(paid.p,0) AS outstanding FROM customers c "
                        "JOIN billed ON billed.customernumber=c.customernumber "
                        "LEFT JOIN paid ON paid.customernumber=c.customernumber "
                        "WHERE billed.b-COALESCE(paid.p,0)>0 ORDER BY outstanding DESC;"},
]
print(f"eval_graph ready · {len(TEST_CASES)} test cases")


In [ ]:
# quick smoke test — run one question through the partial flow
smoke = eval_graph.invoke({"user_input": "how many number of office around the world"})
print("AI SQL     :", smoke.get("output_text2SQL"))
print("Rows       :", smoke.get("execute_sql"))
print("Error round :", smoke.get("sql_retry_count", 0))
print("Run error   :", smoke.get("execute_error"))

AI SQL     : SELECT COUNT(officecode) FROM offices;
Rows       : [(7,)]
Error round : 0
Run error   : None


## Get result 

In [30]:
# run every test case through the partial graph and collect one row per case
rows = []
for case in TEST_CASES:
    q, expected_sql = case["input"], case["expectation_sql"]

    # partial flow: schema -> text2sql -> executeSQL (with self-retry loop)
    final = eval_graph.invoke({"user_input": q})

    actual_sql  = final.get("output_text2SQL")
    actual_out  = final.get("execute_sql")          # list of Row, or None on failure
    error_round = final.get("sql_retry_count", 0)   # how many self-corrections it took
    run_error   = final.get("execute_error")

    expected_out, _ = run_raw_sql(expected_sql)      # ground-truth result

    result_match = same_result(actual_out, expected_out)                 # the real correctness signal
    query_match  = (actual_sql or "").split() == (expected_sql or "").split()  # loose string match (info only)

    log = {
        "input": q,
        "ai_sql": actual_sql,
        "expected_sql": expected_sql,
        "ai_rowcount": None if actual_out is None else len(actual_out),
        "expected_rowcount": None if expected_out is None else len(expected_out),
        "result_match": result_match,
        "error_round": error_round,
        "run_error": run_error,
    }

    rows.append({
        "Input": q,
        "Actual Query": actual_sql,
        "Actual Output": str(actual_out)[:200],
        "Expectation SQL Query": expected_sql,
        "Expectation Output": str(expected_out)[:200],
        "Error Round": error_round,
        # ── eval columns ──
        "AI Query Match": query_match,     # actual vs expected query text (loose)
        "Result Match": result_match,      # actual vs expected fetched data (main metric)
        "Has Error": bool(run_error) or error_round > 0,
        "Log (JSON)": json.dumps(log, ensure_ascii=False, default=str),
    })

print(f"collected {len(rows)} rows")


collected 5 rows


In [43]:
result = []

result.append('3')
result.append('2')

print(result)

['3', '2']


## Pandas 

In [34]:
from pathlib import Path

df = pd.DataFrame(rows)

# quick view (hide the long text columns)
pd.set_option("display.max_colwidth", 50)
print(df[["Input", "Error Round", "AI Query Match", "Result Match", "Has Error"]].to_string(index=False))

# summary metrics
n = len(df)
print(f"\nResult accuracy (fetched data == expectation) : {int(df['Result Match'].sum())}/{n}")
print(f"Query text match                              : {int(df['AI Query Match'].sum())}/{n}")
print(f"Avg error rounds (self-corrections)           : {df['Error Round'].mean():.2f}")

# export to Excel (needs openpyxl: `uv add openpyxl`)
out = Path("..") / "output" / "eval_text2sql.xlsx"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(out, index=False)
print("Saved:", out.resolve())

df


                                                                                 Input  Error Round  AI Query Match  Result Match  Has Error
                                                         How many customers are there?            0           False          True      False
                                How many offices are there and where are they located?            0           False         False      False
                                              What is the total sales revenue in 2004?            0           False          True      False
                                          Total sales revenue by product line in 2004.            0           False          True      False
List customers who exceeded their credit limit based on total orders, and by how much.            0           False         False      False

Result accuracy (fetched data == expectation) : 3/5
Query text match                              : 0/5
Avg error rounds (self-corrections)           : 0

,Input,Actual Query,Actual Output,Expectation SQL Query,Expectation Output,Error Round,AI Query Match,Result Match,Has Error,Log (JSON)
0,How many customers are there?,SELECT COUNT(customernumber) FROM customers;,"[(122,)]",SELECT COUNT(customerNumber) FROM customers;,"[(122,)]",0,False,True,False,"{""input"": ""How many customers are there?"", ""ai..."
1,How many offices are there and where are they ...,"SELECT city, country, COUNT(officecode) OVER (...","[('San Francisco', 'USA', 7), ('Boston', 'USA'...","SELECT officeCode, city, country FROM offices;","[('1', 'San Francisco', 'USA'), ('2', 'Boston'...",0,False,False,False,"{""input"": ""How many offices are there and wher..."
2,What is the total sales revenue in 2004?,SELECT SUM(od.quantityordered * od.priceeach) ...,"[(Decimal('4515905.51'),)]",SELECT SUM(od.quantityOrdered*od.priceEach) FR...,"[(Decimal('4515905.51'),)]",0,False,True,False,"{""input"": ""What is the total sales revenue in ..."
3,Total sales revenue by product line in 2004.,"SELECT p.productline, SUM(od.quantityordered *...","[('Classic Cars', Decimal('1763136.73')), ('Vi...","SELECT p.productLine, SUM(od.quantityOrdered*o...","[('Classic Cars', Decimal('1763136.73')), ('Mo...",0,False,True,False,"{""input"": ""Total sales revenue by product line..."
4,List customers who exceeded their credit limit...,WITH customer_order_totals AS (SELECT o.custom...,"[('Euro+ Shopping Channel', Decimal('227600.00...","WITH t AS (SELECT o.customerNumber, SUM(od.qua...","[('Euro+ Shopping Channel', Decimal('593089.54...",0,False,False,False,"{""input"": ""List customers who exceeded their c..."
